# Evaluate the Step-DPO Adapter on Held-Out CharXiv Questions

This notebook generates paired base/adapter responses for real IDs from the held-out reasoning split and writes structured JSONL output for subsequent judging.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys

import torch
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

model_id = 'Qwen/Qwen2.5-VL-3B-Instruct'
adapter_path = Path(os.environ.get('DPO_ADAPTER_PATH', '/kaggle/working/qwen_vl_step_dpo_adapter'))
if not (adapter_path / 'adapter_config.json').exists():
    raise FileNotFoundError(f'DPO adapter not found at {adapter_path}')
if not torch.cuda.is_available() or torch.cuda.get_device_capability(0)[0] < 7:
    raise RuntimeError('Evaluation requires a T4-class CUDA GPU.')

split_path = Path('data/splits/eval_reasoning_ids.json')
questions_path = Path('data/CharXiv/data/reasoning_val.json')
images_dir = Path('data/CharXiv/images')
with split_path.open(encoding='utf-8') as handle:
    eval_ids = [str(value) for value in json.load(handle)]
with questions_path.open(encoding='utf-8') as handle:
    question_data = json.load(handle)

missing_image_ids = [qid for qid in eval_ids if not list(images_dir.glob(f'{qid}.*'))]
if missing_image_ids:
    subprocess.run(
        [sys.executable, 'scripts/download_images.py', '--ids-file', str(split_path)],
        check=True,
    )

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    dtype=torch.float16,
    attn_implementation='sdpa',
    device_map={'': 0},
    low_cpu_mem_usage=True,
)
processor = AutoProcessor.from_pretrained(model_id)
ft_model = PeftModel.from_pretrained(base_model, str(adapter_path))
ft_model.eval()
print(f'Loaded adapter from {adapter_path} for {len(eval_ids)} held-out questions.')

In [ ]:
def generate_response(model, image_path, question):
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': str(image_path)},
            {
                'type': 'text',
                'text': 'Analyze this chart. Provide step-by-step reasoning and a final answer.\n' + question,
            },
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt',
    ).to('cuda')
    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            use_cache=True,
        )
    generated_ids = generated_ids[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

sample_count = int(os.environ.get('DPO_EVAL_SAMPLES', '5'))
results = []
for question_id in eval_ids[:sample_count]:
    record = question_data.get(question_id)
    if record is None:
        raise KeyError(f'Held-out question {question_id} is missing from reasoning_val.json')
    image_matches = sorted(images_dir.glob(f'{question_id}.*'))
    if not image_matches:
        raise FileNotFoundError(f'No image found for held-out question {question_id}')

    with ft_model.disable_adapter():
        base_response = generate_response(ft_model, image_matches[0], record['query'])
    adapter_response = generate_response(ft_model, image_matches[0], record['query'])
    result = {
        'question_id': question_id,
        'question': record['query'],
        'ground_truth': record['answer'],
        'base_response': base_response,
        'adapter_response': adapter_response,
    }
    results.append(result)
    print(json.dumps(result, ensure_ascii=False, indent=2))

output_path = Path('/kaggle/working/dpo_holdout_generations.jsonl' if Path('/kaggle/working').exists() else 'dpo_holdout_generations.jsonl')
with output_path.open('w', encoding='utf-8') as handle:
    for result in results:
        handle.write(json.dumps(result, ensure_ascii=False) + '\n')
if len(results) != sample_count or any(not row['base_response'] or not row['adapter_response'] for row in results):
    raise AssertionError('Evaluation did not produce both responses for every requested sample.')
print(f'Saved {len(results)} paired holdout generations to {output_path}')